In [44]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_image
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2, InterpolationMode
from PIL import Image

In [10]:

device = "cuda" if torch.cuda.is_available() else "cpu"


In [102]:
# DINO v2 transform: https://github.com/facebookresearch/dinov3
def make_transform(resize_size: int=256):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

root = Path('data')
image_folder = ImageFolder(root / "train_images")


samples = [
    (Path(path), class_idx)
    for path, class_idx in image_folder.samples
]

class_names = image_folder.classes

class ForgeryDataset(Dataset):
    def __init__(self, samples, mask_dir: Path | None=None, size=256):
        self.samples = samples
        self.mask_dir = mask_dir

        self.size = size

        self.image_transforms = make_transform(self.size)

        self.mask_transforms = v2.Compose([
            v2.ToImage(),
            v2.Resize((self.size, self.size), 
            interpolation=InterpolationMode.NEAREST),
        ])
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.image_transforms(image)

        if self.mask_dir is not None and "forged" in img_path.parent.name:
            mask_path = self.mask_dir / img_path.name.replace(".png", ".npy")
            mask = np.load(mask_path)
            mask = self.mask_transforms(mask).squeeze(0).long()
        else:
            mask = None

        return image, mask, label

train_image_folder = ImageFolder(root / "train_images")

samples = [(Path(p), y) for p, y in train_image_folder.samples]

train_dataset = ForgeryDataset(
    samples=samples,
    mask_dir=root / "train_masks",
)

In [103]:
print(train_dataset[5000][0].size())
print(train_dataset[5000][1].size())

torch.Size([3, 256, 256])
torch.Size([320, 256, 256])
